In [1]:
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
df = pd.read_csv("mma_data_wikipedia.csv")

df.head()

,Event,Date,Division,Weightclass,Fight Number,Fighter,Result,Opponent,Method,Round,Time,Location,Venue,Attendance,Url
0,UFC 219: Cyborg vs. Holm,2017-12-30,Women's Featherweight,145.0,0,"Cris ""Cyborg"" Justino",win,Holly Holm,"Decision (unanimous) (49-46, 48-47, 48-47)",5.0,5:00,"Las Vegas, Nevada, U.S.",T-Mobile Arena,13561.0,https://en.wikipedia.org/wiki/UFC_219
1,UFC 219: Cyborg vs. Holm,2017-12-30,Women's Featherweight,145.0,0,Holly Holm,loss,Cris Cyborg,"Decision (unanimous) (49-46, 48-47, 48-47)",5.0,5:00,"Las Vegas, Nevada, U.S.",T-Mobile Arena,13561.0,https://en.wikipedia.org/wiki/UFC_219
2,UFC 219: Cyborg vs. Holm,2017-12-30,Lightweight,155.0,1,Khabib Nurmagomedov,win,Edson Barboza,"Decision (unanimous) (30-25, 30-25, 30-24)",3.0,5:00,"Las Vegas, Nevada, U.S.",T-Mobile Arena,13561.0,https://en.wikipedia.org/wiki/UFC_219
3,UFC 219: Cyborg vs. Holm,2017-12-30,Lightweight,155.0,1,Edson Barboza,loss,Khabib Nurmagomedov,"Decision (unanimous) (30-25, 30-25, 30-24)",3.0,5:00,"Las Vegas, Nevada, U.S.",T-Mobile Arena,13561.0,https://en.wikipedia.org/wiki/UFC_219
4,UFC 219: Cyborg vs. Holm,2017-12-30,Lightweight,155.0,2,Dan Hooker,win,Marc Diakiese,Submission (guillotine choke),3.0,0:42,"Las Vegas, Nevada, U.S.",T-Mobile Arena,13561.0,https://en.wikipedia.org/wiki/UFC_219


In [4]:
df.info()
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 9008 entries, 0 to 9007
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Event         9008 non-null   str    
 1   Date          9008 non-null   str    
 2   Division      8764 non-null   str    
 3   Weightclass   8764 non-null   float64
 4   Fight Number  9008 non-null   int64  
 5   Fighter       9008 non-null   str    
 6   Result        8940 non-null   str    
 7   Opponent      9008 non-null   str    
 8   Method        8942 non-null   str    
 9   Round         8538 non-null   float64
 10  Time          8940 non-null   str    
 11  Location      9008 non-null   str    
 12  Venue         9008 non-null   str    
 13  Attendance    8532 non-null   float64
 14  Url           9008 non-null   str    
dtypes: float64(3), int64(1), str(11)
memory usage: 2.9 MB


Index(['Event', 'Date', 'Division', 'Weightclass', 'Fight Number', 'Fighter',
       'Result', 'Opponent', 'Method', 'Round', 'Time', 'Location', 'Venue',
       'Attendance', 'Url'],
      dtype='str')

In [5]:
df = df.dropna()

df.reset_index(drop=True, inplace=True)

In [6]:
stop_words = set(stopwords.words("english"))

def clean_text(text):

    tokens = word_tokenize(text.lower())

    words = [word for word in tokens if word.isalpha()]

    filtered_words = [word for word in words if word not in stop_words]

    return " ".join(filtered_words)

In [8]:
df.head()

,Event,Date,Division,Weightclass,Fight Number,Fighter,Result,Opponent,Method,Round,Time,Location,Venue,Attendance,Url
0,UFC 219: Cyborg vs. Holm,2017-12-30,Women's Featherweight,145.0,0,"Cris ""Cyborg"" Justino",win,Holly Holm,"Decision (unanimous) (49-46, 48-47, 48-47)",5.0,5:00,"Las Vegas, Nevada, U.S.",T-Mobile Arena,13561.0,https://en.wikipedia.org/wiki/UFC_219
1,UFC 219: Cyborg vs. Holm,2017-12-30,Women's Featherweight,145.0,0,Holly Holm,loss,Cris Cyborg,"Decision (unanimous) (49-46, 48-47, 48-47)",5.0,5:00,"Las Vegas, Nevada, U.S.",T-Mobile Arena,13561.0,https://en.wikipedia.org/wiki/UFC_219
2,UFC 219: Cyborg vs. Holm,2017-12-30,Lightweight,155.0,1,Khabib Nurmagomedov,win,Edson Barboza,"Decision (unanimous) (30-25, 30-25, 30-24)",3.0,5:00,"Las Vegas, Nevada, U.S.",T-Mobile Arena,13561.0,https://en.wikipedia.org/wiki/UFC_219
3,UFC 219: Cyborg vs. Holm,2017-12-30,Lightweight,155.0,1,Edson Barboza,loss,Khabib Nurmagomedov,"Decision (unanimous) (30-25, 30-25, 30-24)",3.0,5:00,"Las Vegas, Nevada, U.S.",T-Mobile Arena,13561.0,https://en.wikipedia.org/wiki/UFC_219
4,UFC 219: Cyborg vs. Holm,2017-12-30,Lightweight,155.0,2,Dan Hooker,win,Marc Diakiese,Submission (guillotine choke),3.0,0:42,"Las Vegas, Nevada, U.S.",T-Mobile Arena,13561.0,https://en.wikipedia.org/wiki/UFC_219


In [10]:
list(df.columns)

['Event',
 'Date',
 'Division',
 'Weightclass',
 'Fight Number',
 'Fighter',
 'Result',
 'Opponent',
 'Method',
 'Round',
 'Time',
 'Location',
 'Venue',
 'Attendance',
 'Url']

In [14]:
print(df.columns)

Index(['Event', 'Date', 'Division', 'Weightclass', 'Fight Number', 'Fighter',
       'Result', 'Opponent', 'Method', 'Round', 'Time', 'Location', 'Venue',
       'Attendance', 'Url'],
      dtype='str')


In [18]:
print(df.columns)

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='str')


In [21]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [26]:
# Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Example dataset (replace this with your dataset)
from sklearn.datasets import load_iris
data = load_iris()

# Create DataFrame
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create the model
model = LogisticRegression(max_iter=200)

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [30]:
df.columns

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='str')